In [1]:
from sentence_transformers import SentenceTransformer, util
from graph.src.db import *
import sys



D:\uit_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('keepitreal/vietnamese-sbert')

In [3]:
mongo_client = init_mongo()
if not mongo_client:
    print("Failed to connect to MongoDB. Exiting.")
    sys.exit()
mongo_db = mongo_client["KB_UIT"]

You successfully connected to MongoDB!


In [4]:
concepts_collection = mongo_db["concepts"]
relations_collection = mongo_db["relations"]

In [5]:
from retrieval.src.db.vector_db import *

In [6]:
db = ConceptRelationDB()

✓ Database setup thành công!


In [7]:
# from typing import List, Dict, Any, Optional, Union
# def to_object_id(value: Union[str, ObjectId]) -> ObjectId:
#     """Convert string hoặc ObjectId sang ObjectId"""
#     if isinstance(value, ObjectId):
#         return value
#     try:
#         return ObjectId(value)
#     except:
#         raise ValueError(f"Invalid ObjectId: {value}")

In [12]:
def search_document_ids(text: str,collection,is_concept: bool = True) -> []:
    r = db.search_concepts(text, top_k=1)[0] if is_concept else db.search_relations(text, top_k=1)[0]
    item = collection.find_one({"_id": to_object_id(r["parent_id"])})
    ids = []
    for doc in item["documents"]:
        ids.append(doc["document_id"])
    return ids

In [8]:
def search_concepts_document_ids(text,concepts_collection) -> []:
    return search_document_ids(text, concepts_collection, is_concept=True)
def search_relations_document_ids(text,relations_collection) -> []:
    return search_document_ids(text, relations_collection, is_concept=False)

In [9]:
r = db.search_concepts("trường đại học", top_k=1)[0]
print(r)

{'id': 346, 'name': 'đại học', 'parent_id': ObjectId('69257aedc6c7e65364bb3935'), 'score': 0.7796}


D:\uit_chatbot\.venv\Lib\site-packages\sentence_transformers\util\tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  a = torch.tensor(a)


In [10]:
search_document_ids("đại học",concepts_collection)

{'id': 346, 'name': 'đại học', 'parent_id': ObjectId('69257aedc6c7e65364bb3935'), 'score': 1.0}


['2b2a2dfc42c7e9e9c6ac886fe42e1143a122a833a62edd244562c76662b45a77']

In [27]:
def search_triplet(triplet) -> []:
    c1 = triplet["c1"]
    c2 = triplet["c2"]
    r = triplet["r"]
    ds1  = search_concepts_document_ids(c1,concepts_collection)
    ds2 = search_concepts_document_ids(c2 ,concepts_collection)
    ds3 = search_relations_document_ids(r, relations_collection)
    set1 = set(ds1)
    set2 = set(ds2)
    set3 = set(ds3)
    # --- 1. Priority 1: Giao của cả 3 (c1 AND c2 AND r) ---
    priority_1_ids_set = set1.intersection(set2).intersection(set3)
    priority_1_ids = list(priority_1_ids_set)

    # --- 2. Priority 2: Giao từng cặp 2 (Loại trừ Priority 1) ---

    # Giao của từng cặp
    pair_c1_c2 = set1.intersection(set2)
    pair_c1_r = set1.intersection(set3)
    pair_c2_r = set2.intersection(set3)

    # Hợp của tất cả các giao cặp
    priority_2_ids_raw = pair_c1_c2.union(pair_c1_r).union(pair_c2_r)

    # Loại trừ những IDs đã thuộc Priority 1
    priority_2_ids_set = priority_2_ids_raw.difference(priority_1_ids_set)
    priority_2_ids = list(priority_2_ids_set)

    # --- 3. Priority 3: Gộp đơn lẻ (Loại trừ Priority 1 và 2) ---

    # Hợp của tất cả các tập hợp (gồm tất cả các IDs đã tìm thấy)
    all_found_ids = set1.union(set2).union(set3)

    # Tập hợp các ID đã được phân loại (Priority 1 và 2)
    classified_ids = priority_1_ids_set.union(priority_2_ids_set)

    # Priority 3 là những ID còn lại (chỉ xuất hiện đơn lẻ)
    priority_3_ids_set = all_found_ids.difference(classified_ids)
    priority_3_ids = list(priority_3_ids_set)

    # --- Kết quả trả về ---
    results = [
        {
            "score": 3,
            "ids": priority_1_ids
        },
        {
            "score": 2,
            "ids": priority_2_ids
        },
        {
            "score": 1,
            "ids": priority_3_ids
        }
    ]

    return results

In [13]:
search_document_ids("áp dụng",relations_collection,False)

{'id': 525, 'name': 'áp dụng', 'parent_id': '69257ae8c6c7e65364bb38da', 'score': 1.0}


['e45f232c967e5c38b036f3bad944ca0c15c2b101423e91ed6139b8f3286f7a2b']

In [10]:
triplet = {
    'c1': "quy chế",
    'c2': "đơn vị",
    'r': "áp dụng với"
}

In [17]:
results = search_triplet(triplet)
for r in results:
    print(r)

{'fcd0071c966d3c3481709ed772930b1d5f59ff45af0873f9de3af95fed92286c', '206a69147716a42aa4d7f122c2db340e4c02cf89bfee7d37b0db1cfc61eccd45', '0888c1089d3dff0e06c23890bcdf71f589ec79413e7fc56cbce888f23e213bbc', '0fd1a8914801b98ab84b729da67b03be3ec6624551dcf3607ffe170184102699', '2bc98fb9f9baac1f9d4dd190441dff9a0f9a8daac2bf40f5d177b60a36cb6430'}
{'fcd0071c966d3c3481709ed772930b1d5f59ff45af0873f9de3af95fed92286c', '9f3f7d2490d201bd5ff39577bf55281c2b183d8dd7b178a09e05679f5dd5a6bc', 'c29536bb87057891703edba893c214f1adb2dfadb57138d814e8c4cf15982efa'}
{'fcd0071c966d3c3481709ed772930b1d5f59ff45af0873f9de3af95fed92286c'}
{'priority': 1, 'ids': ['fcd0071c966d3c3481709ed772930b1d5f59ff45af0873f9de3af95fed92286c']}
{'priority': 2, 'ids': []}
{'priority': 3, 'ids': ['206a69147716a42aa4d7f122c2db340e4c02cf89bfee7d37b0db1cfc61eccd45', '0fd1a8914801b98ab84b729da67b03be3ec6624551dcf3607ffe170184102699', '0888c1089d3dff0e06c23890bcdf71f589ec79413e7fc56cbce888f23e213bbc', '9f3f7d2490d201bd5ff39577bf55281c2b18

In [31]:
from collections import defaultdict


def search_triplets(triplets: list) -> []:
    doc_priority_scores = defaultdict(list)

    # 1. Thực hiện tìm kiếm cho từng triplet
    for triplet in triplets:
        # Gọi hàm tìm kiếm cho từng triplet. Kết quả là danh sách 3 phần tử P1, P2, P3.
        triplet_results = search_triplet(triplet)

        # 2. Xử lý và tích lũy điểm ưu tiên (Priority Score)
        # Gán điểm ưu tiên cho mỗi mức độ: P1=3, P2=2, P3=1

        for result in triplet_results:
            score = result["score"] # 1, 2, hoặc 3
            ids = result["ids"]

            for doc_id in ids:
                doc_priority_scores[doc_id].append(score)

    # 3. Tính tổng điểm và tạo danh sách kết quả cuối cùng
    final_results = []
    for doc_id, scores in doc_priority_scores.items():
        total_score = sum(scores)

        # Có thể tính thêm điểm trung bình (average_score) hoặc điểm max (max_score)
        # để có thêm tiêu chí sắp xếp, nhưng tổng điểm là phổ biến nhất.

        final_results.append({
            "doc_id": doc_id,
            "total_score": total_score,
        })

    final_results.sort(key=lambda x: x["total_score"], reverse=True)

    return final_results

In [1]:
triplets = [
    {
    'c1': "quy chế",
    'c2': "đơn vị",
    'r': "áp dụng với"
    },
    {
    'c1': "quy chế",
    'c2': "cá nhân",
    'r': "áp dụng với"
    },
    {
    'c1': "chương trình",
    'c2': "ngôn ngữ",
    'r': "giảng dạy"
    }
]

In [2]:
from retrieval.src.retrieval.triplet_retriever import TripletRetriever

D:\uit_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
retriever = TripletRetriever()
rs = retriever.search_triplets(triplets)
for r in rs:
    print(r)

You successfully connected to MongoDB!
✓ Database setup thành công!


D:\uit_chatbot\.venv\Lib\site-packages\sentence_transformers\util\tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  a = torch.tensor(a)


[] [] []
[{'score': 3, 'ids': []}, {'score': 2, 'ids': []}, {'score': 1, 'ids': []}]
[] [] []
[{'score': 3, 'ids': []}, {'score': 2, 'ids': []}, {'score': 1, 'ids': []}]
[] [] []
[{'score': 3, 'ids': []}, {'score': 2, 'ids': []}, {'score': 1, 'ids': []}]
